# SKANN-SSL V6 Training — 512 Hz Real HAVS Data
**Architecture V5 (locked)**
- 1D SK Filterbank: kernels (7,15,31,63,127,255,511,1023), 64 filters, same padding
- 2D Backbone: (B,1,64,15360) → L1 3×3 s=1×1 → L2 3×3 s=1×2 → L3 3×3 s=1×2 → L4 3×3 s=2×1 → L5 3×3 s=2×2 → GAP → h(512)
- Projector: 512 → 2048 → 2048 → 256 (BN+ReLU), training only
- Dataset: loads pre-extracted .npy tensors; pairs from pairing_manifest.csv
- Loss: Barlow Twins on z (256-dim); deploy h (512-dim)


## Cell -1: Clear GPU Memory

In [1]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
else:
    print("No GPU — switch runtime to T4")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM free: 101.4 GB


## Cell 0: Installs & Imports

In [2]:
!pip install -q soundfile umap-learn joblib scipy
import torch, numpy as np, pandas as pd, os, time, gc, re, json
from datetime import datetime
from scipy import stats
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

PyTorch 2.10.0+cu128  |  CUDA: True


## Cell 1: Mount Drive & Verify Paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

SSL_ROOT      = '/content/drive/MyDrive/SKANN_SSL'
OUTPUT_DIR    = os.path.join(SSL_ROOT, 'ssl_output_v6')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Manifests ────────────────────────────────────────────────────────────────
PAIRING_MANIFEST = os.path.join(SSL_ROOT, 'ssl_data_50w', 'pairing_manifest.csv')

# ── Tensor directories ───────────────────────────────────────────────────────
TENSOR_VESSEL_DIR = os.path.join(SSL_ROOT, 'v5_data', 'tensors', 'vessel')

# Ambient pools not used in training — paths kept for reference only
# TENSOR_Q_DIR = os.path.join(SSL_ROOT, 'v5_data', 'tensors', 'ambient_Q')
# TENSOR_T_DIR = os.path.join(SSL_ROOT, 'v5_data', 'tensors', 'ambient_T')

for p in [PAIRING_MANIFEST, TENSOR_VESSEL_DIR]:
    assert os.path.exists(p), f'Not found: {p}'

import pandas as pd
pm = pd.read_csv(PAIRING_MANIFEST)
print(f"Pairing manifest : {len(pm):,} pairs")
print(f"  train={(pm['split']=='train').sum():,}  val={(pm['split']=='val').sum():,}")
print(f"  columns: {list(pm.columns)}")
print(f"  K=8 check: {(pm.groupby('anchor').size()==8).all()}")
print(f"Tensor dir       : {TENSOR_VESSEL_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")


Mounted at /content/drive
Pairing manifest : 56,400 pairs
  train=46,400  val=10,000
  columns: ['anchor', 'partner', 'pair_type', 'split', 'feat_cosine_dist', 'rms_ratio', 'gap_sec']
  K=8 check: True
Tensor dir       : /content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel
Output dir       : /content/drive/MyDrive/SKANN_SSL/ssl_output_v6


## Cell 2: Configuration

In [4]:
import os, time
from datetime import datetime

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

# =====================================================================
# HYPERPARAMETERS
# =====================================================================
EPOCHS           = 60
BATCH_SIZE       = 32
BASE_LR          = 3e-5
WEIGHT_DECAY     = 0.01
BT_LAMBDA        = 5e-3
LATENT_DIM       = 256

LOG_EVERY           = 1
HEALTH_CHECK_EVERY  = 5
CHECKPOINT_EVERY    = 5

# =====================================================================
# DATA PARAMETERS
# =====================================================================
FS           = 512
WINDOW_SEC   = 30.0
WINDOW_SAMP  = int(WINDOW_SEC * FS)    # 15360

POOL_T_FRAC  = 0.25

# Dataset virtual length per epoch
# pairing_manifest has ~5900 train pairs; sample 1160 per epoch (10 per event)
TRAIN_PAIRS_PER_EPOCH = 1160
VAL_PAIRS             = 200

# =====================================================================
# AMBIENT MIX-IN
# =====================================================================
AMBIENT_MIX_P      = 0.5
AMBIENT_SNR_RANGE  = (-30, -20)   # dB
MIN_SNR_HEADROOM   = 10.0

print("="*60)
print("SKANN-SSL V5 Configuration")
print("="*60)
print(f"  Window: {WINDOW_SEC}s = {WINDOW_SAMP} samples @ {FS} Hz")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  LR: {BASE_LR}  |  BT_lambda: {BT_LAMBDA}")
print(f"  Latent dim: {LATENT_DIM}")
print(f"  SK Kernels: (7, 15, 31, 63, 127, 255, 511, 1023)")
print(f"  2D backbone: 5× Conv2d BN+ReLU, strides (1,1)→(1,2)→(1,2)→(2,1)→(2,2), GAP → 512")
print(f"  Projector: 512 → 2048 → 2048 → {LATENT_DIM}")
print("="*60)


SKANN-SSL V5 Configuration
  Window: 30.0s = 15360 samples @ 512 Hz
  Batch size: 32
  Epochs: 60
  LR: 3e-05  |  BT_lambda: 0.005
  Latent dim: 256
  SK Kernels: (7, 15, 31, 63, 127, 255, 511, 1023)
  2D backbone: 5× Conv2d BN+ReLU, strides (1,1)→(1,2)→(1,2)→(2,1)→(2,2), GAP → 512
  Projector: 512 → 2048 → 2048 → 256


## Cell 3: Model — HybridSKEncoderV5


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# SKANN-SSL V5 ENCODER  (HybridSKEncoderV5)
#
# Architecture (locked):
#   Input: (B, 1, 15360)  — 30s @ 512 Hz, z-scored
#
#   1D SK Filterbank:
#     8 parallel Conv1d branches, kernels (7,15,31,63,127,255,511,1023)
#     64 filters each, same padding, GELU, GroupNorm
#     SK attention fusion → (B, 64, 15360)
#
#   Reshape: (B, 64, 15360) → (B, 1, 64, 15360)
#     H=64 = SK filter axis, W=15360 = time axis
#
#   2D Backbone (all BN + ReLU):
#     L1: Conv2d  1→ 64, 3×3, s=1×1  →  (B,  64, 64, 15360)
#     L2: Conv2d 64→128, 3×3, s=1×2  →  (B, 128, 64,  7680)
#     L3: Conv2d 128→256,3×3, s=1×2  →  (B, 256, 64,  3840)
#     L4: Conv2d 256→512,3×3, s=2×1  →  (B, 512, 32,  3840)
#     L5: Conv2d 512→512,3×3, s=2×2  →  (B, 512, 16,  1920)
#     AdaptiveAvgPool2d(1,1)          →  (B, 512,  1,     1)
#     Flatten                         →  h (B, 512)  ← DEPLOYMENT
#
#   Projector (training only, discarded after SSL):
#     512 → 2048 (BN+ReLU) → 2048 (BN+ReLU) → 256
#     z (256-dim) → Barlow Twins loss
# =====================================================================

def _norm_1d(channels, groups=8):
    return nn.GroupNorm(min(groups, channels), channels)


class SKConv1D(nn.Module):
    """Selective Kernel 1D Conv — V5 (8 kernels)."""

    def __init__(self, in_ch, out_ch,
                 kernel_sizes=(7, 15, 31, 63, 127, 255, 511, 1023),
                 stride=1, reduction=16, residual=False):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Conv1d(in_ch, out_ch, k, stride=stride,
                      padding=k//2, bias=False)
            for k in kernel_sizes
        ])
        self.n_branches = len(kernel_sizes)
        self.out_ch = out_ch
        hidden = max(out_ch // reduction, 8)
        self.fc1 = nn.Linear(out_ch, hidden)
        self.fc2 = nn.Linear(hidden, out_ch * self.n_branches)
        self.norm = _norm_1d(out_ch)
        self.act  = nn.GELU()
        self.residual = residual
        self.match = None
        if residual and in_ch != out_ch:
            self.match = nn.Conv1d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        feats = [b(x) for b in self.branches]
        U = torch.stack(feats, dim=1).sum(dim=1)
        s = F.adaptive_avg_pool1d(U, 1).squeeze(-1)
        z = self.fc2(F.relu(self.fc1(s), inplace=False))
        a = F.softmax(z.view(z.size(0), self.n_branches, self.out_ch), dim=1).unsqueeze(-1)
        V = (a * torch.stack(feats, dim=1)).sum(dim=1)
        out = self.act(self.norm(V))
        if self.residual:
            out = out + (x if self.match is None else self.match(x))
        return out


class SKFilterbank(nn.Module):
    """SK Filterbank — V5 (8 kernels, 512 Hz)."""

    KERNELS = (7, 15, 31, 63, 127, 255, 511, 1023)

    def __init__(self, out_ch=64):
        super().__init__()
        self.stem = SKConv1D(1, out_ch, self.KERNELS, residual=False)
        self.post_norm = _norm_1d(out_ch)
        print(f"    SKFilterbank kernels : {self.KERNELS}")
        print(f"    Freq coverage @ 512Hz: "
              f"{512/self.KERNELS[-1]:.2f} Hz – {512/self.KERNELS[0]:.1f} Hz")

    def forward(self, x):
        return self.post_norm(self.stem(x))


def _conv2d_bn_relu(in_ch, out_ch, kernel=3, stride=(1,1)):
    pad = (kernel//2, kernel//2)
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=False),
    )


class HybridSKEncoderV5(nn.Module):
    """
    SKANN-SSL V5 Encoder — 512 Hz, locked architecture.
    return_features=False → z  (256-dim, for Barlow Twins loss)
    return_features=True  → (h, z)  (h=512-dim deployment embedding)
    """

    def __init__(self, latent_dim=256):
        super().__init__()
        print("\n" + "="*60)
        print("Building SKANN-SSL V5 Encoder (512 Hz)")
        print("="*60)

        # ── 1D SK front-end ──────────────────────────────────────────
        # Input: (B, 1, 15360)
        # Output: (B, 64, 15360)
        self.sk_frontend = SKFilterbank(out_ch=64)
        # NO AvgPool1d — reshape directly to 2D

        # ── 2D Backbone ──────────────────────────────────────────────
        # Reshape: (B, 64, 15360) → (B, 1, 64, 15360)
        # H=64 (SK filter axis), W=15360 (time axis)
        self.l1 = _conv2d_bn_relu( 1,  64, stride=(1,1))  # → (B,  64, 64, 15360)
        self.l2 = _conv2d_bn_relu(64, 128, stride=(1,2))  # → (B, 128, 64,  7680)
        self.l3 = _conv2d_bn_relu(128,256, stride=(1,2))  # → (B, 256, 64,  3840)
        self.l4 = _conv2d_bn_relu(256,512, stride=(2,1))  # → (B, 512, 32,  3840)
        self.l5 = _conv2d_bn_relu(512,512, stride=(2,2))  # → (B, 512, 16,  1920)
        self.pool = nn.AdaptiveAvgPool2d(1)               # → (B, 512,  1,     1)
        # h: (B, 512)

        # ── Projector ────────────────────────────────────────────────
        # Discarded after SSL training — only h is deployed
        self.projector = nn.Sequential(
            nn.Linear(512,  2048), nn.BatchNorm1d(2048), nn.ReLU(inplace=False),
            nn.Linear(2048, 2048), nn.BatchNorm1d(2048), nn.ReLU(inplace=False),
            nn.Linear(2048, latent_dim),
        )

        n_total = sum(p.numel() for p in self.parameters())
        n_proj  = sum(p.numel() for p in self.projector.parameters())
        n_back  = n_total - n_proj
        print(f"    2D input shape       : (B, 1, 64, 15360)")
        print(f"    L1 s=1×1  →  64×64×15360")
        print(f"    L2 s=1×2  → 128×64×7680")
        print(f"    L3 s=1×2  → 256×64×3840")
        print(f"    L4 s=2×1  → 512×32×3840")
        print(f"    L5 s=2×2  → 512×16×1920")
        print(f"    GAP → h (512-dim)   ← DEPLOYMENT EMBEDDING")
        print(f"    Projector: 512→2048→2048→{latent_dim}  (z, SSL only)")
        print(f"    Backbone params : {n_back/1e6:.2f}M")
        print(f"    Projector params: {n_proj/1e6:.2f}M")
        print(f"    Total params    : {n_total/1e6:.2f}M")
        print("="*60 + "\n")

    def forward(self, x, return_features=False):
        # Ensure (B, 1, T)
        if x.dim() == 1: x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 2: x = x.unsqueeze(1)

        # 1D SK front-end
        x = self.sk_frontend(x)        # (B, 64, 15360)

        # Reshape for 2D backbone — no intermediate downsampling
        x = x.unsqueeze(1)             # (B, 1, 64, 15360)

        # 2D backbone
        x = self.l1(x)                 # (B,  64, 64, 15360)
        x = self.l2(x)                 # (B, 128, 64,  7680)
        x = self.l3(x)                 # (B, 256, 64,  3840)
        x = self.l4(x)                 # (B, 512, 32,  3840)
        x = self.l5(x)                 # (B, 512, 16,  1920)
        h = self.pool(x).flatten(1)    # (B, 512)

        z = self.projector(h)          # (B, latent_dim)

        return (h, z) if return_features else z


# ── Smoke test ───────────────────────────────────────────────────────
model = HybridSKEncoderV5(latent_dim=LATENT_DIM)
dummy = torch.randn(2, 1, WINDOW_SAMP)   # 2 × 1 × 15360
with torch.no_grad():
    h, z = model(dummy, return_features=True)
print(f"Smoke test: input {tuple(dummy.shape)} → h {tuple(h.shape)}, z {tuple(z.shape)}")
assert h.shape == (2, 512),        f'h shape wrong: {h.shape}'
assert z.shape == (2, LATENT_DIM), f'z shape wrong: {z.shape}'
print("Architecture OK")


# =====================================================================
# V6 AUGMENTATION: Random Spectral Tilt
# =====================================================================
# Root cause diagnosis (confirmed V5 post-training):
#   PC1 explains 75% of h-space variance.
#   PC1 correlates with sub-20 Hz energy ratio at Spearman |r|=0.794.
#   The dominant feature is spectral SLOPE (ratio of low-to-high
#   frequency energy), not absolute amplitude.
#
# Why random_gain failed:
#   random_gain multiplies the entire waveform by a scalar.
#   The tonal_score diagnostic measures a frequency RATIO — numerator
#   and denominator scale equally, ratio is unchanged.
#   Spectral slope is invariant to uniform gain. random_gain cannot
#   decorrelate PC1 from spectral slope.
#
# Correct fix — random spectral tilt:
#   In the frequency domain, multiply amplitude at bin k by (f_k)^α
#   where α is drawn independently for anchor and partner from U[−1, +1].
#   α > 0 → blue tilt (more high-frequency energy)
#   α < 0 → red tilt  (more low-frequency energy)
#   α = 0 → flat (no change)
#   Two views of the same event now have different spectral slopes.
#   The encoder cannot use slope to minimise the Barlow Twins loss —
#   it must find slope-invariant structure: harmonic relationships,
#   modulation patterns, temporal envelope shape.
#
# Implementation:
#   1. rfft(x) → complex spectrum
#   2. Build frequency axis normalised to [ε, 1] (avoid zero at DC)
#   3. Multiply each bin by f_norm^α  (real positive scalar per bin)
#   4. irfft back to time domain
#   5. Re-zscore: restore unit variance for consistent encoder input
#      (tilt changes the overall energy level; zscore makes it clean)
#
# α range U[−1, +1] rationale:
#   At f=0.01 (lowest non-DC bin): f^1 = 0.01,  f^-1 = 100  → 40 dB tilt
#   At f=1.0  (Nyquist):           f^1 = 1.0,   f^-1 = 1.0  → 0 dB
#   This spans the range of spectral slopes observed across the 141
#   vessel events without introducing numerical instability.
#
# V6 acceptance criterion (run Spearman diagnostic post-training):
#   PC1 explained variance  < 30%   (vs 75% in V5)
#   Peak Spearman |r|       < 0.40  (vs 0.794 in V5)
#   Tonal |r| during training should decline visibly by epoch 10.
# =====================================================================

def random_spectral_tilt(x: torch.Tensor, alpha_range: float = 1.0) -> torch.Tensor:
    """
    Apply random spectral tilt to a waveform tensor.

    Multiplies the amplitude spectrum by f_norm^α where:
      - f_norm is the normalised frequency axis in [ε, 1]
      - α is drawn independently from U[−alpha_range, +alpha_range]

    After tilting, re-zscores the waveform to restore unit variance.

    Args:
        x           : tensor shape (B, 1, T) or (B, T) or (1, T)
        alpha_range : symmetric tilt range (default ±1.0)
    Returns:
        Tilted, re-zscored tensor, same shape and device as input.
    """
    squeeze_dim = None
    if x.dim() == 3:          # (B, 1, T)
        squeeze_dim = 1
        x = x.squeeze(1)      # → (B, T)
    elif x.dim() == 1:
        x = x.unsqueeze(0)    # → (1, T)

    B, T = x.shape
    device = x.device

    # FFT
    X = torch.fft.rfft(x, n=T)                        # (B, T//2+1) complex

    # Frequency axis normalised to [ε, 1] — avoids zero at DC
    n_bins = X.shape[-1]
    eps    = 1e-3
    f_norm = torch.linspace(eps, 1.0, n_bins, device=device)  # (n_bins,)

    # Independent α per item in batch
    alpha = (torch.rand(B, 1, device=device) * 2 - 1) * alpha_range  # (B, 1)

    # Tilt weights: f_norm^α — computed in log space for numerical stability
    # shape: (B, n_bins)
    tilt = torch.exp(alpha * torch.log(f_norm.unsqueeze(0)))   # (B, n_bins)

    # Apply tilt to amplitude, preserve phase
    X_tilted = X * tilt                                        # (B, n_bins)

    # IFFT back to time domain
    x_out = torch.fft.irfft(X_tilted, n=T)                    # (B, T)

    # Re-zscore: restore unit variance for consistent encoder input
    mu  = x_out.mean(dim=-1, keepdim=True)
    std = x_out.std(dim=-1,  keepdim=True).clamp(min=1e-6)
    x_out = (x_out - mu) / std

    if squeeze_dim is not None:
        x_out = x_out.unsqueeze(squeeze_dim)                   # restore (B,1,T)

    return x_out


print("random_spectral_tilt() defined — V6 augmentation ready")
print("Tilt range: α ~ U[-1.0, +1.0]  |  f_norm axis: [0.001, 1.0]")

# ── Quick sanity check ───────────────────────────────────────────────
import numpy as np
_x_test = torch.randn(4, 15360)
_x_tilt = random_spectral_tilt(_x_test)
assert _x_tilt.shape == _x_test.shape, "Shape mismatch"
assert abs(_x_tilt.std(dim=-1).mean().item() - 1.0) < 0.01, "Zscore failed"
print(f"Sanity check: input std={_x_test.std(dim=-1).mean():.3f}  "
      f"output std={_x_tilt.std(dim=-1).mean():.3f}  ✓")
del _x_test, _x_tilt



Building SKANN-SSL V5 Encoder (512 Hz)
    SKFilterbank kernels : (7, 15, 31, 63, 127, 255, 511, 1023)
    Freq coverage @ 512Hz: 0.50 Hz – 73.1 Hz
    2D input shape       : (B, 1, 64, 15360)
    L1 s=1×1  →  64×64×15360
    L2 s=1×2  → 128×64×7680
    L3 s=1×2  → 256×64×3840
    L4 s=2×1  → 512×32×3840
    L5 s=2×2  → 512×16×1920
    GAP → h (512-dim)   ← DEPLOYMENT EMBEDDING
    Projector: 512→2048→2048→256  (z, SSL only)
    Backbone params : 4.05M
    Projector params: 5.78M
    Total params    : 9.83M

Smoke test: input (2, 1, 15360) → h (2, 512), z (2, 256)
Architecture OK
random_spectral_tilt() defined — V6 augmentation ready
Tilt range: α ~ U[-1.0, +1.0]  |  f_norm axis: [0.001, 1.0]
Sanity check: input std=1.000  output std=1.000  ✓


## Cell 5: Dataset — Event-Aware SSL
ShuffleQueue for Pool T diversity. CPA bias 35%. Minimum pair separation 90s.

In [6]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# SKANNSSLDataset V5
# Loads anchor + partner .npy tensors directly from pairing_manifest.
# No ambient pool loading. No augmentation. Tensors loaded as-is.
# =====================================================================

def _load_tensor(path):
    """Load a .npy tensor. Returns float32 array of shape (15360,)."""
    x = np.load(path).astype(np.float32)
    if x.ndim > 1:
        x = x.flatten()
    if len(x) < WINDOW_SAMP:
        x = np.pad(x, (0, WINDOW_SAMP - len(x)))
    return x[:WINDOW_SAMP]


class SKANNSSLDataset(Dataset):
    """
    Barlow Twins dataset V5.
    Each __getitem__: sample one row from pairing_manifest, load
    anchor.npy and partner.npy, return raw float32 tensors.
    No augmentation. No ambient mix-in.
    """

    def __init__(self, pairing_manifest, tensor_dir, split,
                 dataset_len=TRAIN_PAIRS_PER_EPOCH):
        self.tensor_dir = tensor_dir
        self.dlen       = dataset_len

        pm = pd.read_csv(pairing_manifest)
        self.pairs = pm[pm['split'] == split].reset_index(drop=True)
        print(f"[{split}] {len(self.pairs):,} pairs in manifest "
              f"(sampling {dataset_len} per epoch)")

    def __len__(self):
        return self.dlen

    def __getitem__(self, _):
        row = self.pairs.iloc[np.random.randint(len(self.pairs))]
        x1  = _load_tensor(os.path.join(self.tensor_dir, row['anchor']))
        x2  = _load_tensor(os.path.join(self.tensor_dir, row['partner']))
        return torch.from_numpy(x1).float(), torch.from_numpy(x2).float()


print('SKANNSSLDataset V5 defined')

# ── Instantiation check ───────────────────────────────────────────────────────
train_ds = SKANNSSLDataset(PAIRING_MANIFEST, TENSOR_VESSEL_DIR, 'train',
                           dataset_len=TRAIN_PAIRS_PER_EPOCH)
val_ds   = SKANNSSLDataset(PAIRING_MANIFEST, TENSOR_VESSEL_DIR, 'val',
                           dataset_len=VAL_PAIRS)
v1, v2 = train_ds[0]
print(f"Sample pair: v1={tuple(v1.shape)} v2={tuple(v2.shape)} dtype={v1.dtype}")


SKANNSSLDataset V5 defined
[train] 46,400 pairs in manifest (sampling 1160 per epoch)
[val] 10,000 pairs in manifest (sampling 200 per epoch)
Sample pair: v1=(15360,) v2=(15360,) dtype=torch.float32


## Cell 6: Loss & Diagnostics
Barlow Twins (unchanged). New: 51-53 Hz Spearman correlation + same/diff event cosine similarity.

In [7]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy import stats

# =====================================================================
# BARLOW TWINS LOSS  (unchanged)
# =====================================================================

def barlow_twins_loss(z1, z2, lambd=BT_LAMBDA):
    B   = z1.size(0)
    z1n = (z1 - z1.mean(0)) / (z1.std(0) + 1e-6)
    z2n = (z2 - z2.mean(0)) / (z2.std(0) + 1e-6)
    c   = torch.mm(z1n.T, z2n) / B
    d   = torch.diagonal(c)
    on  = torch.pow(1.0 - d, 2).sum()
    off = (torch.pow(c, 2).sum() - torch.pow(d, 2).sum())
    return on + lambd * off


# =====================================================================
# DIAGNOSTIC: 51-53 Hz TONAL CORRELATION  (Spearman, val set)
# Target: |r| < 0.2 by epoch 10.
# V5: reads .npy tensors from vessel_window_features paths.
# =====================================================================

def compute_tonal_score(x_np, fs=512, f_low=0.5, f_high=20.0,
                         f_ref_low=20.0, f_ref_high=200.0):
    # V6 UPDATE: track 0.5–20 Hz sub-band energy ratio.
    # V5 post-training diagnostic confirmed PC1 (75% variance) correlates
    # with sub-20 Hz energy at Spearman |r|=0.794 — NOT the 50-54 Hz band
    # which was the original hypothesis (|r| only 0.235 there).
    # Target for V6: |r| < 0.4 by epoch 10 with random_spectral_tilt augmentation.
    n   = len(x_np)
    X   = np.abs(np.fft.rfft(x_np))**2
    f   = np.fft.rfftfreq(n, 1.0/fs)
    band = X[(f >= f_low)  & (f <= f_high)].mean()  + 1e-12
    ref  = X[(f >= f_ref_low) & (f <= f_ref_high)].mean() + 1e-12
    return float(band / ref)


@torch.no_grad()
def compute_diagnostics(model, val_ds, device):
    """
    Runs diagnostics on val pairs.
    Returns dict: tonal_spearman_r, same_event_sim, diff_event_sim,
                  sim_gap, sil_h_event, n_val_windows.
    """
    model.eval()

    pm = pd.read_csv(PAIRING_MANIFEST)
    WINDOW_POOL_MANIFEST = os.path.join(SSL_ROOT, 'ssl_data_50w', 'window_pool_50.csv')
    # window_pool_50.csv provides tensor_file → event_id mapping
    wm = pd.read_csv(WINDOW_POOL_MANIFEST)[['tensor_file', 'event_id']].drop_duplicates()
    tf_to_eid = dict(zip(wm['tensor_file'], wm['event_id']))
    val_pairs = pm[pm['split'] == 'val'].reset_index(drop=True)

    # Sample up to 400 val pairs for speed
    n_sample = min(400, len(val_pairs))
    idx = np.random.choice(len(val_pairs), n_sample, replace=False)
    val_pairs = val_pairs.iloc[idx].reset_index(drop=True)

    h_all, tonal_scores, event_ids = [], [], []
    same_sims, diff_sims = [], []

    for _, row in val_pairs.iterrows():
        for col in ['anchor', 'partner']:
            try:
                x = np.load(os.path.join(TENSOR_VESSEL_DIR,
                                         row[col])).astype(np.float32)
                x = x.flatten()[:WINDOW_SAMP]
            except:
                continue
            ts = compute_tonal_score(x)
            xt = torch.from_numpy(x).float().unsqueeze(0).to(device)
            h, _ = model(xt, return_features=True)
            h_np = h.cpu().numpy().squeeze()
            h_all.append(h_np)
            tonal_scores.append(ts)
            event_ids.append(tf_to_eid.get(row[col], -1))

        # Same-event similarity: anchor vs positive within this pair
        if len(h_all) >= 2:
            a, b = h_all[-2], h_all[-1]
            if event_ids[-2] == event_ids[-1]:  # intra-event
                sim = np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8)
                same_sims.append(sim)

    h_all = np.array(h_all)
    tonal_scores = np.array(tonal_scores)
    event_ids = np.array(event_ids)

    # Cross-event similarity
    for _ in range(min(200, len(h_all))):
        i, j = np.random.choice(len(h_all), 2, replace=False)
        if event_ids[i] != event_ids[j]:
            a, b = h_all[i], h_all[j]
            diff_sims.append(
                np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8))

    # Tonal Spearman correlation
    pc1 = PCA(n_components=1).fit_transform(h_all).squeeze()
    r_pos, _ = stats.spearmanr(tonal_scores, pc1)
    r_neg, _ = stats.spearmanr(tonal_scores, -pc1)
    tonal_r  = r_pos if abs(r_pos) > abs(r_neg) else r_neg

    sil_h = -1.0
    if len(np.unique(event_ids)) > 1:
        try:
            sil_h = float(silhouette_score(h_all, event_ids, metric='cosine'))
        except: pass

    return {
        'tonal_spearman_r': float(tonal_r),
        'same_event_sim':   float(np.mean(same_sims)) if same_sims else 0.0,
        'diff_event_sim':   float(np.mean(diff_sims)) if diff_sims else 0.0,
        'sim_gap':          float(np.mean(same_sims) - np.mean(diff_sims))
                            if same_sims and diff_sims else 0.0,
        'sil_h_event':      sil_h,
        'n_val_windows':    len(h_all),
    }

print("Loss and diagnostic functions defined (V5)")


Loss and diagnostic functions defined (V5)


## Cell 7: Training Loop
Pool T queue reset every epoch. Diagnostic interpretation with actionable warnings.

In [8]:
import time, os, sys
import torch
from torch.utils.data import DataLoader

def train():
    # ── Device ──────────────────────────────────────────────────────
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

    # ── Model ───────────────────────────────────────────────────────
    model = HybridSKEncoderV5(latent_dim=LATENT_DIM).to(device)

    # ── DataLoader ──────────────────────────────────────────────────
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2,
                              pin_memory=True, drop_last=True)
    n_batches = len(train_loader)

    # ── Optimiser / Scheduler / AMP ─────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler()

    # ── Log file ────────────────────────────────────────────────────
    loss_csv = os.path.join(OUTPUT_DIR, 'loss_history.csv')
    with open(loss_csv, 'w') as f:
        f.write("epoch,loss,lr,tonal_r,same_sim,diff_sim,sim_gap,sil_h_event\n")

    # ── Dashboard helpers ────────────────────────────────────────────
    W = 62  # box width

    def box_top(title=''):
        if title:
            pad = W - 4 - len(title)
            print(f"\n┌── {title} {'─'*pad}┐")
        else:
            print(f"\n┌{'─'*(W)}┐")

    def box_row(left, right=''):
        content = f"  {left:<{W-4}}{right}"
        print(f"│{content[:W]}│")

    def box_mid():
        print(f"├{'─'*W}┤")

    def box_bot():
        print(f"└{'─'*W}┘")

    def bar(frac, width=30):
        filled = int(frac * width)
        return '█' * filled + '░' * (width - filled)

    def fmt_time(secs):
        secs = int(secs)
        h, m, s = secs//3600, (secs%3600)//60, secs%60
        return (f"{h}h {m:02d}m {s:02d}s" if h else f"{m}m {s:02d}s")

    # ── Header ──────────────────────────────────────────────────────
    box_top('SKANN-SSL V6  512 Hz Training  [+spectral_tilt]')
    box_row(f"GPU   : {gpu_name}")
    box_row(f"VRAM  : {vram_gb:.1f} GB  |  AMP: ON  |  Batch: {BATCH_SIZE}")
    box_row(f"Epochs: {EPOCHS}  |  Batches/epoch: {n_batches}  |  LR: {BASE_LR}")
    n_total = sum(p.numel() for p in model.parameters())/1e6
    n_proj  = sum(p.numel() for p in model.projector.parameters())/1e6
    box_row(f"Params: {n_total:.2f}M total  ({n_total - n_proj:.2f}M backbone)")
    box_bot()

    best_same_sim = -1.0
    t_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        t_epoch = time.time()

        for batch_idx, (v1, v2) in enumerate(train_loader):
            v1 = v1.to(device, non_blocking=True)
            v2 = v2.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                # V6: independent spectral tilt per view — breaks slope consistency
                z1 = model(random_spectral_tilt(v1))
                z2 = model(random_spectral_tilt(v2))
                loss = barlow_twins_loss(z1, z2)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

            # Inline batch progress
            frac = (batch_idx + 1) / n_batches
            elapsed = time.time() - t_epoch
            eta_batch = elapsed / (batch_idx + 1) * (n_batches - batch_idx - 1)
            print(f"\r  Epoch {epoch:02d}/{EPOCHS} │ "
                  f"Batch {batch_idx+1:03d}/{n_batches} │ "
                  f"[{bar(frac, 20)}] {frac*100:5.1f}% │ "
                  f"Loss {loss.item():.4f} │ "
                  f"ETA {fmt_time(eta_batch)}   ",
                  end='', flush=True)

        print()  # newline after batch bar
        scheduler.step()
        avg_loss   = total_loss / n_batches
        current_lr = scheduler.get_last_lr()[0]
        elapsed    = time.time() - t_start
        eta_train  = elapsed / epoch * (EPOCHS - epoch)
        epoch_time = time.time() - t_epoch

        # ── Health diagnostics ───────────────────────────────────────
        diag = None
        if epoch % HEALTH_CHECK_EVERY == 0:
            diag = compute_diagnostics(model, val_ds, device)
            sim_gap = diag['same_event_sim'] - diag['diff_event_sim']

            if diag['same_event_sim'] > best_same_sim:
                best_same_sim = diag['same_event_sim']
                torch.save({
                    'epoch': epoch,
                    'encoder': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(),
                    'loss': avg_loss,
                    'best_same_sim': best_same_sim,
                }, os.path.join(OUTPUT_DIR, 'best_model.pth'))

        # ── Epoch dashboard ──────────────────────────────────────────
        ep_frac = epoch / EPOCHS
        box_top(f'Epoch {epoch:02d}/{EPOCHS}')
        box_row(f"Progress  [{bar(ep_frac, 40)}] {ep_frac*100:.0f}%")
        box_mid()
        box_row(f"Loss      : {avg_loss:.6f}",
                f"  LR : {current_lr:.2e}")
        box_row(f"Epoch time: {fmt_time(epoch_time)}",
                f"  ETA: {fmt_time(eta_train)}")
        if diag:
            tr = abs(diag['tonal_spearman_r'])
            sim_gap = diag['same_event_sim'] - diag['diff_event_sim']
            box_mid()
            box_row(f"Tonal |r| : {tr:.3f}  (target < 0.2)")
            box_row(f"Same sim  : {diag['same_event_sim']:.4f}  "
                    f"Diff sim: {diag['diff_event_sim']:.4f}  "
                    f"Gap: {sim_gap:.4f}")
            box_row(f"Sil event : {diag['sil_h_event']:.4f}")
            # Warnings
            if tr > 0.4:
                box_row(f"⚠  TONAL LEAK |r|={tr:.3f} — increase pool_T_frac")
            if sim_gap < 0.05:
                box_row(f"⚠  WEAK STRUCTURE gap={sim_gap:.4f} — check LR/batch")
            if diag['same_event_sim'] > 0.5 and tr < 0.2:
                box_row(f"✓  On track: structure forming, tonal leak suppressed")
        box_bot()

        # ── CSV log ──────────────────────────────────────────────────
        with open(loss_csv, 'a') as f:
            if diag:
                tr = diag['tonal_spearman_r']
                sim_gap = diag['same_event_sim'] - diag['diff_event_sim']
                f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},"
                        f"{tr:.4f},{diag['same_event_sim']:.4f},"
                        f"{diag['diff_event_sim']:.4f},{sim_gap:.4f},"
                        f"{diag['sil_h_event']:.4f}\n")
            else:
                f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},,,,\n")

        # ── Checkpoint every 5 epochs ────────────────────────────────
        if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
            ckpt = os.path.join(OUTPUT_DIR, f'BT_ckpt_epoch_{epoch:03d}.pth')
            torch.save({
                'epoch': epoch,
                'encoder': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(),
                'loss': avg_loss,
                'best_same_sim': best_same_sim,
            }, ckpt)
            print(f"  💾 Checkpoint saved: BT_ckpt_epoch_{epoch:03d}.pth")

    # ── Final save ───────────────────────────────────────────────────
    final = os.path.join(OUTPUT_DIR, 'SKANN_SSL_V6_Final.pth')
    torch.save(model.state_dict(), final)

    box_top('Training Complete')
    box_row(f"Final model : SKANN_SSL_V6_Final.pth")
    box_row(f"Best same-event sim: {best_same_sim:.4f}")
    box_row(f"Total time  : {fmt_time(time.time() - t_start)}")
    box_bot()
    return model

model = train()



Building SKANN-SSL V5 Encoder (512 Hz)
    SKFilterbank kernels : (7, 15, 31, 63, 127, 255, 511, 1023)
    Freq coverage @ 512Hz: 0.50 Hz – 73.1 Hz
    2D input shape       : (B, 1, 64, 15360)
    L1 s=1×1  →  64×64×15360
    L2 s=1×2  → 128×64×7680
    L3 s=1×2  → 256×64×3840
    L4 s=2×1  → 512×32×3840
    L5 s=2×2  → 512×16×1920
    GAP → h (512-dim)   ← DEPLOYMENT EMBEDDING
    Projector: 512→2048→2048→256  (z, SSL only)
    Backbone params : 4.05M
    Projector params: 5.78M
    Total params    : 9.83M



/tmp/ipykernel_1968/829806579.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



┌── SKANN-SSL V6  512 Hz Training  [+spectral_tilt] ───────────┐
│  GPU   : NVIDIA RTX PRO 6000 Blackwell Server Edition      │
│  VRAM  : 102.0 GB  |  AMP: ON  |  Batch: 32                │
│  Epochs: 60  |  Batches/epoch: 36  |  LR: 3e-05            │
│  Params: 9.83M total  (4.05M backbone)                     │
└──────────────────────────────────────────────────────────────┘


/tmp/ipykernel_1968/829806579.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Epoch 01/60 │ Batch 036/36 │ [████████████████████] 100.0% │ Loss 171.4498 │ ETA 0m 00s   

┌── Epoch 01/60 ───────────────────────────────────────────────┐
│  Progress  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2%   │
├──────────────────────────────────────────────────────────────┤
│  Loss      : 182.335278                                      │
│  Epoch time: 5m 49s                                          │
└──────────────────────────────────────────────────────────────┘
  Epoch 02/60 │ Batch 036/36 │ [████████████████████] 100.0% │ Loss 119.6228 │ ETA 0m 00s   

┌── Epoch 02/60 ───────────────────────────────────────────────┐
│  Progress  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3%   │
├──────────────────────────────────────────────────────────────┤
│  Loss      : 137.390289                                      │
│  Epoch time: 3m 33s                                          │
└──────────────────────────────────────────────────────────────┘
  Epoch 03/60 │ Batch 036/36 │ [████

...

/tmp/ipykernel_1968/829806579.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Epoch 21/60 │ Batch 036/36 │ [████████████████████] 100.0% │ Loss 60.1087 │ ETA 0m 00s   

┌── Epoch 21/60 ───────────────────────────────────────────────┐
│  Progress  [██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░] 35%  │
├──────────────────────────────────────────────────────────────┤
│  Loss      : 62.601502                                       │
│  Epoch time: 0m 56s                                          │
└──────────────────────────────────────────────────────────────┘
  Epoch 22/60 │ Batch 036/36 │ [████████████████████] 100.0% │ Loss 53.5452 │ ETA 0m 00s   

┌── Epoch 22/60 ───────────────────────────────────────────────┐
│  Progress  [██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░] 37%  │
├──────────────────────────────────────────────────────────────┤
│  Loss      : 60.334047                                       │
│  Epoch time: 0m 56s                                          │
└──────────────────────────────────────────────────────────────┘
  Epoch 23/60 │ Batch 036/36 │ [██████

/tmp/ipykernel_1968/829806579.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Epoch 26/60 │ Batch 014/36 │ [███████░░░░░░░░░░░░░]  38.9% │ Loss 48.8753 │ ETA 0m 34s   

KeyboardInterrupt: 

## Cell 8: Resume from Checkpoint (optional)

In [ ]:
# Run this cell INSTEAD of Cell 7 to resume training
ckpts = sorted(__import__('glob').glob(os.path.join(OUTPUT_DIR, 'BT_ckpt_epoch_*.pth')))
if not ckpts:
    print("No checkpoint found — run Cell 7 to train from scratch")
else:
    ckpt_path = ckpts[-1]
    ckpt = torch.load(ckpt_path, map_location='cpu')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = HybridSKEncoderV5(latent_dim=LATENT_DIM).to(device)
    model.load_state_dict(ckpt['encoder'])
    log(f"Resumed from {ckpt_path}  epoch={ckpt['epoch']}  "
        f"loss={ckpt['loss']:.4f}  best_sim={ckpt.get('best_same_sim',-1):.4f}")
    # Then call model = train() — scheduler/optimizer will restart from scratch
    # For full resume including optimizer state, extend train() to accept ckpt arg

In [ ]:
from tqdm import tqdm
WINDOW_POOL_MANIFEST = os.path.join(SSL_ROOT, 'ssl_data_50w', 'window_pool_50.csv')

## Cell 9: Extract Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings_from_clips(model, split='all'):
    """
    Extract h and z embeddings for all vessel clips.
    Uses centre window of each clip for a single representative embedding per event.
    For visualisation and clustering — not training.
    """
    log(f"Extracting embeddings (split={split})...")
    device = next(model.parameters()).device
    model.eval()

    # Use window_pool_50.csv — one row per tensor, has event_id + split + assessment
    vm = pd.read_csv(WINDOW_POOL_MANIFEST)
    if split != 'all':
        vm = vm[vm['split'] == split]
    vm = vm.drop_duplicates(subset='event_id').reset_index(drop=True)

    h_all, z_all, event_ids, assessments = [], [], [], []

    for _, row in tqdm(vm.iterrows(), total=len(vm), desc="Extracting"):
        # V5: tensors are fixed 30s windows; use vessel_window_features for multi-window
        # V5: load pre-extracted .npy tensor (centre window = the tensor itself)
        tensor_file = row['tensor_file']
        try:
            x = np.load(os.path.join(TENSOR_VESSEL_DIR, tensor_file)).astype(np.float32)
            x = x.flatten()[:WINDOW_SAMP]
            if len(x) < WINDOW_SAMP: x = np.pad(x, (0, WINDOW_SAMP-len(x)))
        except:
            x = np.zeros(WINDOW_SAMP, dtype=np.float32)

        xt = torch.from_numpy(x[:WINDOW_SAMP]).float().unsqueeze(0).to(device)
        h, z = model(xt, return_features=True)
        h_all.append(h.cpu().numpy().squeeze())
        z_all.append(z.cpu().numpy().squeeze())
        event_ids.append(row['event_id'])
        assessments.append(row['assessment'])

    h_all = np.array(h_all)
    z_all = np.array(z_all)

    np.save(os.path.join(OUTPUT_DIR, f'embeddings_h_{split}.npy'), h_all)
    np.save(os.path.join(OUTPUT_DIR, f'embeddings_z_{split}.npy'), z_all)

    log(f"h: {h_all.shape}  z: {z_all.shape}")
    return h_all, z_all, np.array(event_ids), assessments

h_embeddings, z_embeddings, event_ids, assessments = extract_embeddings_from_clips(model, 'all')

## Cell 10: Cluster & Visualise
K-Means on h (k=5). UMAP coloured by cluster and by HAVS assessment.
**Semi-supervised step**: inspect clusters, then assign vessel class labels manually.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import umap

# =====================================================================
# CLUSTERING — semi-supervised step
# Cluster h embeddings → assign pseudo-labels → validate with GT
# =====================================================================

N_CLUSTERS = 5   # tanker, cargo, fishing, small_craft, no_vessel

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(h_embeddings)

# Map GT assessments to colour
assess_colors = {
    'CONFIRMED VESSEL': '#E31A1C',
    'PROBABLE VESSEL':  '#FF7F00',
}

# UMAP for visualisation
reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', random_state=42)
h_2d = reducer.fit_transform(h_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: coloured by cluster
ax = axes[0]
cmap = plt.colormaps['tab10'].resampled(N_CLUSTERS)
for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    ax.scatter(h_2d[mask,0], h_2d[mask,1], c=[cmap(k)],
               label=f'Cluster {k} (n={mask.sum()})', s=50, alpha=0.7,
               edgecolors='white', linewidth=0.4)
ax.set_title(f'UMAP of h — K-Means clusters (k={N_CLUSTERS})')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: coloured by HAVS assessment
ax = axes[1]
colors_assess = [assess_colors.get(a, '#999999') for a in assessments]
ax.scatter(h_2d[:,0], h_2d[:,1], c=colors_assess, s=50, alpha=0.7,
           edgecolors='white', linewidth=0.4)
from matplotlib.patches import Patch
patches = [Patch(color=v, label=k) for k,v in assess_colors.items()]
ax.legend(handles=patches, loc='upper right', fontsize=8)
ax.set_title('UMAP coloured by HAVS assessment')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'umap_clusters.png'), dpi=300, bbox_inches='tight')
plt.show()

# Cluster composition summary
vm = pd.read_csv(WINDOW_POOL_MANIFEST).drop_duplicates(subset='event_id').reset_index(drop=True)
vm['cluster'] = cluster_labels
print("\nCluster composition:")
print(vm.groupby(['cluster','assessment']).size().unstack(fill_value=0).to_string())

In [ ]:
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial import Voronoi, voronoi_plot_2d
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import numpy as np

# =====================================================================
# EXTENDED VISUALISATION
# 1. Silhouette plot (per-sample + per-cluster + overall)
# 2. t-SNE (coloured by cluster and by assessment)
# 3. Voronoi territory map with centroids
# =====================================================================

cmap = plt.colormaps['tab10'].resampled(N_CLUSTERS)
cluster_colors = [cmap(k) for k in range(N_CLUSTERS)]

# ── 1. Silhouette Plot ────────────────────────────────────────────────────────
sil_overall = silhouette_score(h_embeddings, cluster_labels, metric='cosine')
sil_samples = silhouette_samples(h_embeddings, cluster_labels, metric='cosine')

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
sil_cluster_means = []

for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    vals = np.sort(sil_samples[mask])
    sil_cluster_means.append(vals.mean())
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=cluster_colors[k], edgecolor='none', alpha=0.85)
    ax.text(-0.05, y_lower + len(vals) / 2,
            f'C{k}\n(n={mask.sum()})', ha='right', va='center', fontsize=8)
    y_lower = y_upper + 10

ax.axvline(x=sil_overall, color='crimson', linestyle='--', linewidth=1.5,
           label=f'Overall silhouette = {sil_overall:.4f}')
for k, m in enumerate(sil_cluster_means):
    ax.axvline(x=m, color=cluster_colors[k], linestyle=':', linewidth=1,
               alpha=0.7, label=f'C{k} mean = {m:.3f}')

ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster samples (sorted)')
ax.set_title(f'Silhouette Analysis — K-Means k={N_CLUSTERS} on h embeddings (cosine)')
ax.legend(loc='lower right', fontsize=7)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'silhouette_plot.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Overall silhouette score : {sil_overall:.4f}')
for k, m in enumerate(sil_cluster_means):
    print(f'  Cluster {k} mean silhouette : {m:.4f}  (n={( cluster_labels==k).sum()})')

# ── 2. t-SNE ─────────────────────────────────────────────────────────────────
print('\nRunning t-SNE...')
tsne = TSNE(n_components=2, perplexity=min(30, len(h_embeddings)//3),
            metric='cosine', random_state=42, n_iter=1000, verbose=0)
h_tsne = tsne.fit_transform(h_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# t-SNE coloured by cluster
ax = axes[0]
for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    ax.scatter(h_tsne[mask, 0], h_tsne[mask, 1],
               c=[cluster_colors[k]], label=f'Cluster {k} (n={mask.sum()})',
               s=60, alpha=0.8, edgecolors='white', linewidth=0.4)
ax.set_title('t-SNE of h — K-Means clusters')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

# t-SNE coloured by HAVS assessment
ax = axes[1]
assess_colors = {'CONFIRMED VESSEL': '#E31A1C', 'PROBABLE VESSEL': '#FF7F00'}
colors_assess = [assess_colors.get(a, '#999999') for a in assessments]
ax.scatter(h_tsne[:, 0], h_tsne[:, 1], c=colors_assess,
           s=60, alpha=0.8, edgecolors='white', linewidth=0.4)
patches = [mpatches.Patch(color=v, label=k) for k, v in assess_colors.items()]
patches.append(mpatches.Patch(color='#999999', label='other'))
ax.legend(handles=patches, loc='upper right', fontsize=8)
ax.set_title('t-SNE coloured by HAVS assessment')
ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_clusters.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── 3. Voronoi Territory Map (on UMAP projection) ────────────────────────────
# Project K-Means centroids into UMAP space via nearest-neighbour approximation
# (centroids are in 512-dim h space; find closest embedded point per cluster)
print('\nBuilding Voronoi territory map...')

# Find UMAP representative point for each cluster centroid
centroid_2d = np.zeros((N_CLUSTERS, 2))
for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    # Centroid in 2D = mean of UMAP coords for cluster members
    centroid_2d[k] = h_2d[mask].mean(axis=0)

fig, ax = plt.subplots(figsize=(12, 9))

# Scatter all points
for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    ax.scatter(h_2d[mask, 0], h_2d[mask, 1],
               c=[cluster_colors[k]], s=50, alpha=0.6,
               edgecolors='white', linewidth=0.3,
               label=f'Cluster {k} (n={mask.sum()})')

# Voronoi diagram from centroids
# Add mirror points at large distance to bound the diagram
pad = 10.0
x_min, x_max = h_2d[:, 0].min() - pad, h_2d[:, 0].max() + pad
y_min, y_max = h_2d[:, 1].min() - pad, h_2d[:, 1].max() + pad
mirror_pts = np.array([
    [x_min - 100, 0], [x_max + 100, 0],
    [0, y_min - 100], [0, y_max + 100]
])
vor_pts = np.vstack([centroid_2d, mirror_pts])

try:
    vor = Voronoi(vor_pts)
    for simplex in vor.ridge_vertices:
        simplex = np.asarray(simplex)
        if np.all(simplex >= 0):
            ax.plot(vor.vertices[simplex, 0], vor.vertices[simplex, 1],
                    'k-', linewidth=1.0, alpha=0.5)
except Exception as e:
    print(f'  Voronoi failed: {e} — skipping boundaries')

# Plot centroids
for k in range(N_CLUSTERS):
    ax.scatter(centroid_2d[k, 0], centroid_2d[k, 1],
               c=[cluster_colors[k]], s=250, marker='*',
               edgecolors='black', linewidth=1.2, zorder=5)
    ax.annotate(f'C{k}', centroid_2d[k],
                fontsize=10, fontweight='bold',
                xytext=(4, 4), textcoords='offset points')

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_title(f'Voronoi Territory Map — K-Means centroids on UMAP projection\n'
             f'Overall silhouette = {sil_overall:.4f}')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.2)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'voronoi_territory.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f'\nSaved figures to {OUTPUT_DIR}:')
print('  silhouette_plot.png')
print('  tsne_clusters.png')
print('  voronoi_territory.png')


In [ ]:
# ── t-SNE single panel: clusters + centroids ─────────────────────────────────
# Reuses h_tsne already computed — no re-running t-SNE.

fig, ax = plt.subplots(figsize=(14, 10))

for k in range(N_CLUSTERS):
    mask = cluster_labels == k
    ax.scatter(h_tsne[mask, 0], h_tsne[mask, 1],
               c=[cluster_colors[k]],
               label=f'Cluster {k}  (n={mask.sum()})',
               s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

for k in range(N_CLUSTERS):
    ax.scatter(tsne_centroids[k, 0], tsne_centroids[k, 1],
               c=[cluster_colors[k]], s=350, marker='*',
               edgecolors='black', linewidth=1.4, zorder=5)
    ax.annotate(f'C{k}', tsne_centroids[k],
                fontsize=12, fontweight='bold',
                xytext=(6, 5), textcoords='offset points')

ax.set_title('t-SNE of h embeddings — K-Means clusters with centroids\n'
             f'SKANN-SSL V5  |  512 Hz  |  k={N_CLUSTERS}  |  '
             f'Silhouette = {sil_overall:.4f}',
             fontsize=13, pad=12)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_final.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: tsne_final.png')


## Cell 11: Export Production Bundle

In [ ]:
import joblib

# =====================================================================
# EXPORT PRODUCTION BUNDLE  (h embeddings + model weights + metadata)
# =====================================================================

def export_bundle():
    log("Exporting production bundle...")

    weights = os.path.join(OUTPUT_DIR, "best_model.pth")
    if not os.path.exists(weights):
        weights = os.path.join(OUTPUT_DIR, "SKANN_SSL_V5_Final.pth")

    state = torch.load(weights, map_location='cpu')
    if 'encoder' in state: state = state['encoder']
    state = {k.replace('module.',''): v for k, v in state.items()}

    bundle = {
        'model_state':   state,
        'embeddings_h':  h_embeddings,
        'embeddings_z':  z_embeddings,
        'event_ids':     event_ids,
        'assessments':   assessments,
        'cluster_labels': cluster_labels,
        'metrics': {
            'n_events':    len(h_embeddings),
            'n_clusters':  N_CLUSTERS,
        },
        'metadata': {
            'version':        'v5.0.0-512hz',
            'architecture':   'HybridSKEncoderV5 (SKANN)',
            'sk_kernels':     (7, 15, 31, 63, 127, 255, 511, 1023),
            'projector':      '512→2048→2048→256',
            'latent_dim':     LATENT_DIM,
            'backbone_dim':   512,
            'fs':             FS,
            'window_sec':     WINDOW_SEC,
            'window_samp':    WINDOW_SAMP,
            'n_train_events': int((pd.read_csv(WINDOW_POOL_MANIFEST)['split']=='train').sum() // 50),
            'n_val_events':   int((pd.read_csv(WINDOW_POOL_MANIFEST)['split']=='val').sum() // 50),
            'val_dates':      '2026-02-07 to 2026-02-09',
            'site':           'NUWR Goa, 30m depth, Arabian Sea',
            'export_date':    datetime.now().isoformat(),
        }
    }

    path = os.path.join(OUTPUT_DIR, 'SKANN_SSL_V5_Production_Bundle.joblib')
    joblib.dump(bundle, path, compress=3)

    print("\n" + "="*60)
    print("Production Bundle V4 — 512 Hz")
    print("="*60)
    print(f"  SK kernels: {bundle['metadata']['sk_kernels']}")
    print(f"  @ 512 Hz: {FS/1023:.2f} Hz – {FS/7:.1f} Hz")
    print(f"  2D backbone: (1,2)→(1,2)→Inception+(1,2)→(2,1) strides")
    print(f"  Backbone dim (h): 512  ← DEPLOYMENT EMBEDDING")
    print(f"  Train events: {bundle['metadata']['n_train_events']}")
    print(f"  Val events:   {bundle['metadata']['n_val_events']}")
    print(f"  Size: {os.path.getsize(path)/1e6:.1f} MB")
    print(f"  Path: {path}")
    print("="*60)

export_bundle()

## Cell 12: Training Interpretation Guide

| Diagnostic | Healthy | Warning | Action |
|---|---|---|---|
| Tonal \|r\| | < 0.2 by ep 10 | > 0.4 | Increase pool_T_frac or relax RMS gate to P20 |
| same_event_sim | Rising → > 0.5 | Flat | Check LR, increase BATCH_SIZE |
| diff_event_sim | Low / stable | Rising fast | Augmentation too weak — add more gain jitter |
| sim_gap | > 0.15 by ep 20 | < 0.05 | Architecture / LR issue |
| BT loss | Decreasing | Plateau early | Reduce BT_LAMBDA to 1e-3 |

**Note on labels:** This notebook produces unlabelled clusters. After inspecting the UMAP, assign vessel class labels to clusters using:
- Blade rate from HAVS `transit_events_292.csv`
- GT knowledge (fishing=BR 23Hz, tanker=BR 6.5Hz, ship=BR 47Hz)
- D6-dominant events → small craft / OBM